<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 4: Case D and Multiple Objectives

**A study plan can reduce the learning gap, but doing so usually uses more time.**

Part 3 classified decisions by their allowed values. This part formulates the study-time case from 01-2 and distinguishes raw performance, a scalar objective, and a vector objective.

### 1 · Carry forward the study decision

Let \(m\) and \(w\) be mathematics and writing study hours. The student has at most 4 hours. Two performance outputs are

> $\displaystyle Q(m,w)=\frac{20}{1+m}+\frac{15}{1+w},\qquad S(m,w)=m+w.$

The learning gap \(Q\) becomes smaller with study. The time use \(S\) becomes larger. The same decision can therefore improve one output and worsen the other.

### 2 · Write the multi-objective formulation

Let \(x=[m,w]^{\mathsf T}\) and \(y=\operatorname{Sim}(x)=[Q(x),S(x)]^{\mathsf T}\). Here \(\operatorname{Sim}\) is a direct algebraic response calculation.

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad
\boldsymbol f(y)=\begin{bmatrix}Q(x)\\S(x)\end{bmatrix}$
>
> $\displaystyle \text{subject to}\quad -m\le0,\quad -w\le0,\quad m+w-4\le0.$

Uppercase \(F\) is not used for the vector objective. The notation \(\boldsymbol f\) distinguishes several objective values from the scalar \(f\).

A feasible plan \(x^A\) **dominates** another feasible plan \(x^B\) when it is no worse in both objectives and strictly better in at least one. A Pareto candidate is not dominated by another feasible candidate.

### 3 · Inspect a finite performance set

The next cells define the responses and inspect a 0.25-hour decision grid. The plot tells us to compare learning gap horizontally and study time vertically. Dark teal points are nondominated grid candidates; light teal points are other feasible grid candidates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MAX_STUDY_TIME = 4.0


def evaluate_study_plan(x):
    math_hours, writing_hours = np.asarray(x, dtype=float)
    learning_gap = 20.0 / (1.0 + math_hours) + 15.0 / (1.0 + writing_hours)
    study_time = math_hours + writing_hours
    residuals = np.array([-math_hours, -writing_hours, study_time - MAX_STUDY_TIME])
    return {
        "x": (math_hours, writing_hours),
        "y": (learning_gap, study_time),
        "g": residuals,
        "feasible": bool(np.all(residuals <= 1e-10)),
    }


def pareto_front(records):
    feasible = [record for record in records if record["feasible"]]
    nondominated = []
    for candidate in feasible:
        q_value, s_value = candidate["y"]
        dominated = any(
            other["y"][0] <= q_value
            and other["y"][1] <= s_value
            and (other["y"][0] < q_value or other["y"][1] < s_value)
            for other in feasible
        )
        if not dominated:
            nondominated.append(candidate)
    return sorted(nondominated, key=lambda record: record["y"][1])

In [ ]:
levels = np.arange(0.0, MAX_STUDY_TIME + 0.125, 0.25)
records = [evaluate_study_plan([m, w]) for m in levels for w in levels]
feasible = [record for record in records if record["feasible"]]
pareto = pareto_front(records)

figure, axis = plt.subplots(figsize=(7.2, 4.8))
axis.scatter(
    [record["y"][0] for record in feasible],
    [record["y"][1] for record in feasible],
    color="#80cbc4",
    alpha=0.65,
    s=34,
    label="Feasible grid candidate",
)
axis.plot(
    [record["y"][0] for record in pareto],
    [record["y"][1] for record in pareto],
    color="teal",
    marker="o",
    markersize=4,
    linewidth=2,
    label="Nondominated grid candidates",
)
axis.set(
    xlabel="Learning gap Q",
    ylabel="Study time S (h)",
    title="Less learning gap usually requires more study time",
)
axis.grid(alpha=0.25)
axis.legend(fontsize=8)
figure.tight_layout()
plt.show()
plt.close(figure)

print(f"{len(feasible)} feasible grid candidates; {len(pareto)} nondominated grid candidates")

The teal curve is the Pareto front of the stated finite grid, not a proof of the continuous Pareto front. Moving along it exchanges learning gap for study time. The vector objective identifies trade-offs but does not select one plan by itself.

### 4 · Add a stated selection rule when one plan is needed

A weighted sum creates a scalar objective:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad
f(y;\alpha)=\alpha\,\widetilde Q(x)+(1-\alpha)\,\widetilde S(x),
\qquad 0\le\alpha\le1.$

The tildes indicate normalized values. Normalization prevents hours or the numerical scale of \(Q\) from silently dominating the score. The weight \(\alpha\) is a hyperparameter; it changes the comparison rule, not the outcomes of a fixed study plan.

An alternative keeps one objective and turns the other into a requirement:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad Q(x)
\qquad\text{subject to}\qquad S(x)\le\varepsilon.$

Changing \(\alpha\) changes the scalar score. Changing \(\varepsilon\) changes the feasible set. These are different formulations.

### 5 · Classify Case D

Case D is a **generally constrained, continuous, multi-objective, nonlinear, direct algebraic, deterministic optimization problem**. Choosing a weighted sum would change only its objective classification to single-objective.

### Takeaway

Count the values that the formulation asks us to optimize:

> **one scalar \(f\) → single-objective · vector \(\boldsymbol f\) → multi-objective · dominance removes inferior choices · a preference or requirement selects one trade-off**

Performance outputs describe a plan. The objective supplies the comparison rule. Part 5 keeps these roles visible while changing function structure, response evaluation, and treatment of uncertain weather.